# Week 22, Day 1: Installing Python

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w022d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys
print(sys.version_info[:2] >= (3, 10))
print(sys.version_info.major)
print("3.12" > "3.9", (3, 12) > (3, 9))

In [ ]:
def which(command, path, programs):
    """The first folder on path that contains the command, as the shell does."""
    for folder in path.split(":"):
        if command in programs.get(folder, set()):
            return folder + "/" + command
    return None

PATH = "/opt/homebrew/bin:/usr/local/bin:/usr/bin:/bin"
programs = {"/usr/bin": {"python3", "git", "ls"}, "/opt/homebrew/bin": {"python3", "brew"}, "/bin": {"ls", "cat"}}
print(which("python3", PATH, programs))
print(which("git", PATH, programs))
print(which("pip", PATH, programs))

In [ ]:
import sys, platform

print(sys.version.split()[0])
print(sys.version_info >= (3, 10))
print(sys.executable)                 # the Python program's file (in the browser, an internal name)
print(platform.system(), platform.python_implementation())

In [ ]:
import sys

MINIMUM = (3, 10)
if sys.version_info < MINIMUM:
    sys.exit(f"This program needs Python {MINIMUM[0]}.{MINIMUM[1]} or newer; you have {sys.version.split()[0]}")
print("version is fine")

> **Key idea.** Install a recent Python 3 from python.org (tick "Add to PATH" on Windows), then check `python3 --version` (or `py --version`) in a new terminal. The shell finds commands by searching the folders in `PATH` in order. In code, `sys.version_info` gives the version as comparable numbers and `sys.executable` says which Python is running.

### Exercise: Compare versions properly

Write `parse_version(text)` turning `"3.12.4"` into the tuple `(3, 12, 4)` (a missing bugfix number counts as 0: `"3.9"` is `(3, 9, 0)`), and `newest(versions)` returning the newest version **text** from a list. Comparing the texts directly gets it wrong.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert parse_version(\"3.12.4\") == (3, 12, 4) and parse_version(\"3.9\") == (3, 9, 0)\n    assert newest([\"3.9.18\", \"3.12.4\", \"3.10.14\", \"3.12.10\"]) == \"3.12.10\"\n    assert max([\"3.9\", \"3.12\"]) == \"3.9\", \"this is why text comparison fails\"\n    assert newest([\"3.13.0\"]) == \"3.13.0\"")

### Exercise: Still supported?

A minor version is released in October of its year and supported for 5 years after that. Write `support_ends(minor)` for a version like `"3.12"`, returning the year its support ends, using the table `RELEASED` in the starter, and `supported(minor, year)` returning `True` if it's still supported during that year (up to and including the end year). Return `None` from `support_ends` for a version not in the table.

In [ ]:
RELEASED = {"3.8": 2019, "3.9": 2020, "3.10": 2021, "3.11": 2022, "3.12": 2023, "3.13": 2024, "3.14": 2025}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert support_ends(\"3.12\") == 2028 and support_ends(\"3.8\") == 2024 and support_ends(\"2.7\") is None\n    assert supported(\"3.10\", 2026) is True and supported(\"3.9\", 2026) is False and supported(\"3.9\", 2025) is True\n    assert supported(\"2.7\", 2010) is False")

### Exercise: which

Write `which(command, path, programs, sep=":")` returning the full path of the first folder on `path` (folders separated by `sep`) that contains the command, or `None`; `programs` maps each folder to a set of program names. Then `all_matches(...)` with the same arguments returning every match in order, which is what `which -a` shows when several copies are installed.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    PATH = \"/opt/homebrew/bin:/usr/local/bin:/usr/bin:/bin\"\n    progs = {\"/usr/bin\": {\"python3\", \"git\"}, \"/opt/homebrew/bin\": {\"python3\"}, \"/bin\": {\"ls\"}}\n    assert which(\"python3\", PATH, progs) == \"/opt/homebrew/bin/python3\"\n    assert which(\"ls\", PATH, progs) == \"/bin/ls\" and which(\"pip\", PATH, progs) is None\n    assert all_matches(\"python3\", PATH, progs) == [\"/opt/homebrew/bin/python3\", \"/usr/bin/python3\"]\n    win = r\"C:\\Python312;C:\\Windows\"\n    assert which(\"python.exe\", win, {r\"C:\\Python312\": {\"python.exe\"}}, sep=\";\") == r\"C:\\Python312/python.exe\"")

### Exercise: Which version do I need?

Some features need a minimum Python version. Using the table in the starter, write `minimum_version(features)` returning the lowest `(major, minor)` that has **all** the listed features (the largest of their minimums; `(3, 8)` if the list is empty), and `can_run(features, version_info)` returning `True` if a running version (a tuple such as `(3, 11, 4)`) is new enough.

In [ ]:
NEEDS = {
    "f-strings": (3, 6),
    "walrus": (3, 8),
    "match": (3, 10),
    "union types": (3, 10),
    "exception groups": (3, 11),
    "type statement": (3, 12),
}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert minimum_version([\"f-strings\", \"match\"]) == (3, 10)\n    assert minimum_version([]) == (3, 8) and minimum_version([\"type statement\", \"walrus\"]) == (3, 12)\n    assert can_run([\"match\"], (3, 11, 4)) and not can_run([\"exception groups\"], (3, 10, 9))")

### Exercise: An environment report

Write `report()` returning a list of three strings about the Python running it: `"version: X.Y.Z"` (from `sys.version_info`), `"implementation: NAME"` (from `platform.python_implementation()`), and `"new enough: True"` or `False` for whether it's at least 3.10. This is the kind of report bug trackers ask for.

In [ ]:
import sys
import platform

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"matches this Python\"\n    import sys, platform\n    r = report()\n    v = sys.version_info\n    assert r[0] == f\"version: {v.major}.{v.minor}.{v.micro}\"\n    assert r[1] == f\"implementation: {platform.python_implementation()}\"\n    assert r[2] == f\"new enough: {v >= (3, 10)}\" and len(r) == 3")

### Bug hunt: 3.9 is not newer than 3.10

This check compares versions as text, so it wrongly says 3.9 is new enough. Fix `new_enough` so it prints `False True`.

In [ ]:
def new_enough(version, minimum="3.10"):
    return version >= minimum

print(new_enough("3.9"), new_enough("3.12"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"False True\")\n    assert new_enough(\"3.10\") is True and new_enough(\"3.10.1\") is True")

### Bug hunt: A confusing crash

This script uses a `match` statement, so on Python older than 3.10 it fails with a confusing `SyntaxError` before any of it runs: Python reads the whole file before running a line, so a version check in the same file never gets the chance. Replace the `match` with an equivalent `if` statement, so any Python 3 can read the file, and add a check of `sys.version_info` that exits with a clear message below 3.8. It should print `ok`.

In [ ]:
import sys

command = "go"
match command:
    case "go":
        print("ok")
    case _:
        print("unknown")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ok\"\n    expect_output(\"ok\")\n    assert \"match \" not in source and \"sys.version_info\" in source")

### Puzzle: Check before you run

Arrange the lines so the script stops with a clear message on Python older than 3.10, and otherwise prints the version. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the minor version in "3.12.4"
b = ___   # True or False: "3.9" > "3.12" as strings
c = ___   # the environment variable the shell searches for commands
d = ___   # the Windows command that launches the newest installed Python (two letters)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 12 and b is True and c == \"PATH\" and d == \"py\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: requirements that fit

Projects often say which Python versions they work with, written like `">=3.10,<3.14"`. Write:

- `parse_spec(spec)`: a list of `(operator, version_tuple)` pairs from a comma-separated spec, with operators `>=`, `>`, `<=`, `<`, `==` and `!=` (versions padded to three numbers as in the first exercise; a spec may be empty).
- `allows(spec, version)`: `True` if a version text satisfies every part.
- `best_choice(spec, available)`: the newest version from a list of available version texts that the spec allows, or `None`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert parse_spec(\">=3.10,<3.14\") == [(\">=\", (3, 10, 0)), (\"<\", (3, 14, 0))]\n    assert parse_spec(\"\") == []\n    assert allows(\">=3.10,<3.14\", \"3.12.4\") and not allows(\">=3.10,<3.14\", \"3.14.0\") and not allows(\">=3.10\", \"3.9.18\")\n    assert allows(\"!=3.11.2\", \"3.11.3\") and not allows(\"==3.12\", \"3.12.1\") and allows(\"\", \"3.7\")\n    avail = [\"3.9.18\", \"3.10.14\", \"3.12.4\", \"3.13.1\", \"3.14.0\"]\n    assert best_choice(\">=3.10,<3.14\", avail) == \"3.13.1\"\n    assert best_choice(\">3.14\", avail) is None")

## Warm-up

`sys.version_info` tells a program which version of Python is running it, as numbers that compare properly: `(3, 12)` is bigger than `(3, 9)`, while the **text** `"3.12"` sorts before `"3.9"`. Programs use it to check that the features they need are there.

## Versions

A Python version looks like **3.12.4**: major version 3, **minor** version 12, **bugfix** release 4.

- Python 2 ended in 2020. Everything now is Python 3; any guide that says `print "hello"` without brackets is out of date.
- A new **minor** version comes out every October (3.12 in 2023, 3.13 in 2024, 3.14 in 2025), adding features. Each is supported with bug and security fixes for about five years.
- **Bugfix** releases (3.12.1, 3.12.2, ...) only fix problems; always take the newest for your minor version.

Which to install? The **newest version that's been out for a few months**, so that the libraries you'll use have caught up with it. Avoid "pre-release" (alpha, beta, release candidate) versions for everyday work. This course works on any version from 3.10 upwards; the in-browser Python is a recent version.

New versions mostly **add** things: code written for 3.10 runs on 3.13. The reverse isn't true: `match` statements need 3.10, the `|` in type hints needs 3.10, and some error messages and modules are only in newer versions.

## Installing

Follow the steps for your system. Afterwards, **open a new terminal** (a terminal only reads its settings when it starts) and check with the version command.

**Windows**

1. Go to python.org, Downloads, and get the latest Python 3 installer for Windows.
2. Run it, and on the first screen **tick "Add python.exe to PATH"**. This is the step most people miss.
3. Choose "Install Now".
4. In a new terminal: `py --version` and `python --version`. Windows has the `py` launcher, which picks the newest installed Python: `py script.py` runs a script, and `py -3.12 script.py` asks for a specific version.

If typing `python` opens the Microsoft Store instead, that's a Windows shortcut pretending to be Python: turn it off in Settings, Apps, Advanced app settings, App execution aliases.

**macOS**

1. Go to python.org and download the macOS installer, and run it.
2. In a new Terminal: `python3 --version`. On macOS the command is `python3` (and `pip3`); plain `python` may not exist.
3. Many developers use Homebrew (brew.sh) instead: `brew install python`. Either is fine; just don't mix several ways of installing without knowing which you're using.

**Linux**

Most distributions already have Python 3: try `python3 --version`. That copy belongs to the **system** (the operating system's own tools use it), so don't change or remove it, and don't install packages into it with `sudo pip`. Install a newer version with your package manager (`sudo apt install python3.12` on Ubuntu, perhaps from the "deadsnakes" archive) or with a version manager.

**Version managers.** Tools like `pyenv` or `uv` install several Python versions side by side and switch between them per project. You don't need one yet, but it's worth knowing they exist when a project needs a version you don't have.

## How the terminal finds Python

When you type a command such as `python3`, the shell looks for a program with that name in a list of folders called **`PATH`** (an environment variable, from Week 21), in order, and runs the **first** it finds. "Command not found" means no folder on `PATH` has it.

```
echo $PATH          (macOS and Linux: folders separated by :)
$env:PATH           (PowerShell: folders separated by ;)
which python3       (macOS and Linux: where the command was found)
where.exe python    (Windows)
```

That's why ticking "Add to PATH" matters on Windows: it adds Python's folder to the list. And it explains a common puzzle: with two Pythons installed, the one in the folder that comes **earlier** in `PATH` wins.

## Which Python is this?

Inside a running program, `sys` says exactly which Python is running it:

`sys.executable` is the first thing to print when a package "isn't found" even though you installed it: it usually means it was installed into a **different** Python from the one running your script. Tomorrow's virtual environments solve that for good.

**Checking the version in code:** if a script needs a newer feature, check at the start and fail with a clear message instead of a confusing error later:

## Keeping it tidy

- Install one Python from one source, and know which it is.
- Never install packages into the system's own Python on macOS or Linux, and never with `sudo pip`.
- Use a separate **virtual environment** for each project (tomorrow), so projects can't break each other.
- When something's odd, ask: which Python is running (`sys.executable`), and which `pip` am I using (`pip --version` says which Python it belongs to)?

## Practice

## Quiz

## Daily challenge

## Before you go

Install Python on your computer if you haven't, then in a new terminal run `python3 -c "import sys; print(sys.version, sys.executable)"` (or `py -c ...` on Windows) and note which Python you have and where it lives. Tomorrow: virtual environments and `pip`, so every project gets its own tidy set of packages.